# 18 · Demo de Landsat: cómo se deriva la pérdida de bosque, paso a paso

**Qué hace.** En una zona piloto de Caquetá, entre Cartagena del Chairá y San Vicente del Caguán, parte de
imágenes Landsat 8 y 9 crudas y detecta la pérdida de bosque con el mismo tipo de cálculo que usan los
productos publicados. Después la compara con las alertas GLAD-L y mide dónde y por qué se aparta.

**Para qué sirve.** Es parte del entendimiento de los datos: el panel usa cifras que calcularon otros
(GLAD-L, Hansen, el IDEAM), y la demo replica el tipo de cálculo con que se obtienen, para entenderlo y no
tomarlo como dado. **No alimenta ninguna tabla de deforestación ni ningún modelo.** La explicación completa,
con cada decisión y cada cifra, está en `guias/4_DEMO_LANDSAT.md` del repositorio.

**Qué lee.** No lee tablas de BigQuery. Usa:

| Insumo | De dónde sale | Llave |
|---|---|---|
| Escenas Landsat 8 y 9 | Catálogo público de Microsoft Planetary Computer; se lee solo el recorte de la zona | Ninguna |
| Bosque inicial | El cuadro de Hansen que cubre la zona (se baja en la sección 5) | Ninguna |
| Deforestación del IDEAM 2021-2022 | La capa del IDEAM (se baja en la sección 5) | Ninguna |
| GLAD-L | Ráster `date_conf` de Global Forest Watch | La API key de GFW |

**Qué escribe.** Nueve tablas en `DATASET_RESULTADOS`, que empiezan por `DEMO_LANDSAT_` (sección 17).

**Qué tienes que cambiar.** Solo la API key de GFW, en la celda 1; lo demás ya apunta al proyecto del
Observatorio (`prueba-ofr`).

**Cuánto tarda.** Unos 20 minutos la primera vez, casi todo leyendo unas 70 escenas Landsat. Si se vuelve a
correr en la misma máquina, los compuestos quedan guardados y tarda segundos.

**El método.** Un píxel cuenta como pérdida si cumple tres condiciones:

1. era bosque al inicio, según Hansen (dosel de al menos 30 %);
2. su índice NBR cayó más que un umbral entre la temporada seca de 2022 y la de 2023;
3. pertenece a un parche de al menos 1 ha.

**Qué dio en el trabajo de grado**, sobre la mitad de la zona que no se usó para calibrar:

| Métrica | Valor |
|---|---|
| Concordancia de Lin, hectáreas por celda de 5 km | 0,902 |
| Correlación de Pearson, por celda | 0,909 |
| Precisión, a nivel de píxel | 0,549 |
| Sensibilidad, a nivel de píxel | 0,467 |
| F1, a nivel de píxel | 0,505 |

GLAD-L, Hansen y el IDEAM concuerdan entre sí, en 2022, con un F1 por píxel de 0,44 a 0,61: la réplica cae
dentro de ese rango. GLAD-L es un producto "rodante" que GFW actualiza, así que una corrida nueva puede dar
cifras un poco distintas.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr), salvo
#    la API key de GFW: pégala abajo, donde dice <<LLENAR>>.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset donde quedan los resultados de la segmentación y de los
# modelos. Es el mismo de arriba. Para tenerlos aparte, pon otro
# nombre: el primer notebook que escriba ahí crea el dataset.
DATASET_RESULTADOS = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Llave de Global Forest Watch -------------------------------------
# La misma del notebook 03. Con ella se baja GLAD-L. Vence al año; el
# anexo del notebook 03 dice cómo sacar una nueva. Este notebook queda
# guardado con la llave adentro: no lo compartas fuera del equipo.
GFW_API_KEY = "<<LLENAR: API key de Global Forest Watch>>"

# --- Insumos de la zona (los mismos de los notebooks 02 y 04) ------------
HANSEN_VERSION = "GFC-2025-v1.13"
CAPA_IDEAM_2021_2022 = "cambio_2021_2022_v8_230705.img"

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

# Carpeta donde se bajan los archivos mientras corre el notebook.
# Vacío: una carpeta temporal. Si ya los tienes bajados, apúntala ahí
# y no se vuelven a descargar.
CARPETA_DESCARGAS = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet rasterio==1.4.3 scipy folium==0.14.0 matplotlib requests pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "18_demo_landsat"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

## 4. Los parámetros de la demo

Todo lo que cambia el resultado está aquí, como en el resto del proyecto:

- `BBOX`: la zona piloto. Se escogió porque concentra pérdida suficiente; en una zona estable casi todo sería
  verdadero negativo y las métricas no distinguirían nada.
- `T1` y `T2`: **la misma temporada seca en años consecutivos** (enero a marzo de 2022 y de 2023). Comparar
  enero con julio mediría el ciclo de la vegetación, no la tala. Se usa 2022-2023 porque el ráster de GLAD-L
  que publica GFW arranca en 2021 y no conserva los años anteriores.
- `GLAD_DESDE` y `GLAD_HASTA`: la ventana equivalente para las alertas, del centro de T1 al centro de T2.
- `UMBRAL_DOSEL = 30`: el mismo umbral del panel nacional, para hablar de "bosque" en los mismos términos.
- `UMBRALES_DNBR`: los candidatos que prueba la calibración. Llega hasta 0,70 para que el óptimo quede dentro
  del rango y no en su borde.
- `PIXELES_MIN_PARCHE = 12`: a 30 m cada píxel es 0,09 ha; 12 píxeles (1,08 ha) es el primer entero que llega a
  la hectárea, el área mínima de la definición de bosque del IDEAM.

Los mensajes de la corrida salen con el registro de Python (`logger`), igual que en el módulo del repositorio.

In [ ]:
import datetime as dt
import logging
import os
import time
import warnings
from typing import Dict, List, Optional, Tuple

import rasterio
import requests
from rasterio.transform import Affine
from rasterio.warp import Resampling, reproject, transform_bounds
from rasterio.windows import Window, from_bounds
from scipy import ndimage

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logger = logging.getLogger("demo_landsat")

# Zona piloto: nucleo de deforestacion del arco amazonico, entre
# Cartagena del Chaira y San Vicente del Caguan (Caqueta). Se escoge
# porque concentra perdida suficiente para que las metricas tengan
# sentido; en una zona estable casi todo seria "verdadero negativo" y la
# comparacion no distinguiria nada.
BBOX = (-75.0, 0.8, -74.2, 1.5)          # lon_min, lat_min, lon_max, lat_max

# Ventanas de composicion: misma temporada seca en anios consecutivos.
# Ver "POR QUE LA VENTANA ES 2022-2023" en el encabezado.
T1_INICIO, T1_FIN = "2022-01-01", "2022-03-31"
T2_INICIO, T2_FIN = "2023-01-01", "2023-03-31"

# Ventana equivalente para GLAD-L: del centro de T1 al centro de T2. Un
# compuesto de mediana sobre enero-marzo representa, en promedio, algo
# cercano a la mitad de la ventana; la fecha que representa de verdad
# cada pixel se mide en cada corrida (fecha_efectiva.csv). Darle a cada
# pixel su propia ventana de alertas segun esa fecha se probo y no
# mejoro la concordancia (F1 0,491 frente a 0,505), asi que la ventana
# se deja fija.
GLAD_DESDE = dt.date(2022, 2, 15)
GLAD_HASTA = dt.date(2023, 2, 15)
GLAD_CONF_MIN = 3                        # 3 = alerta confirmada

# Anio calendario en que se compara a los productos profesionales entre
# si (ver referencia_entre_productos). Debe estar cubierto por el
# raster rodante de GLAD-L y tener capa del IDEAM en disco.
ANIO_REFERENCIA = 2022

# Bosque inicial, con los gránulos de Hansen de la zona (sección 5).
UMBRAL_DOSEL = 30                        # mismo valor que config_local.py
ANIO_CORTE_PERDIDA = 21                  # lossyear <= 21 = perdida previa a 2022

# Deteccion propia. La rejilla llega hasta 0,70: con el tope en 0,35 el F1
# todavia venia subiendo en el ultimo valor probado, asi que el optimo
# quedaba fuera del rango y el umbral elegido era el borde de la busqueda.
UMBRALES_DNBR = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35,
                 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

# Area minima de parche. A 30 m cada pixel son 900 m2 = 0,09 ha, asi que
# 11 pixeles son 0,99 ha: se quedan JUSTO por debajo de la hectarea. 12
# es el primer entero que la alcanza (1,08 ha), y es el que se usa para
# no admitir parches menores al minimo del IDEAM.
PIXELES_MIN_PARCHE = 12

ESCALA_M = 30
CRS = "EPSG:32618"                       # UTM 18N, la zona del sur de Colombia
LADO_CELDA_M = 5000                      # celda de agregacion, escala del modelo
FILAS_POR_FRANJA = 512                   # ver compuesto_nbr(): acota la memoria
PUNTOS_POR_ESTRATO = 50
SEMILLA = 42

# Catalogo STAC publico de Microsoft Planetary Computer.
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
SAS_URL = "https://planetarycomputer.microsoft.com/api/sas/v1/token/landsat-c2-l2"
COLECCION_STAC = "landsat-c2-l2"
PLATAFORMAS = ["landsat-8", "landsat-9"]

# GLAD-L en el data-lake de GFW.
GFW_API = "https://data-api.globalforestwatch.org"
GLAD_DATASET = "umd_glad_landsat_alerts"
GLAD_TILE = "10N_080W"                   # tesela de 10x10 que cubre la zona
GLAD_EPOCA = dt.date(2014, 12, 31)       # origen de la codificacion de fecha

DIR_DEMO = DIR_TRABAJO / "demo"
DIR_DEMO.mkdir(parents=True, exist_ok=True)
TIEMPO_ESPERA = 300

## 5. Insumos: Hansen e IDEAM de la zona

Se baja el cuadro de Hansen de 10 × 10 grados que cubre la zona (`10N_080W`), con sus dos capas, y la capa de
cambio de bosque del IDEAM de 2021-2022, que se usa en la sección 13 para comparar los productos entre sí. Los
archivos que ya están en la carpeta de trabajo no se vuelven a bajar.

`reproyectar_sobre_bloque()` es la misma función de los notebooks 02 y 04: lleva un ráster a la rejilla de
trabajo con vecino más cercano.

In [ ]:
DIR_HANSEN = DIR_TRABAJO / "hansen"
DIR_HANSEN.mkdir(parents=True, exist_ok=True)


def cuadros_hansen(bbox):
    """Nombres de los cuadros de Hansen de 10 x 10 grados que tocan el rectángulo."""
    lon_min, lat_min, lon_max, lat_max = bbox
    nombres = []
    for arriba in range(int((lat_max // 10 + 1) * 10), int(lat_min // 10 * 10), -10):
        for izquierda in range(int(lon_min // 10 * 10), int((lon_max // 10 + 1) * 10), 10):
            nombres.append(f"{abs(arriba):02d}{'N' if arriba >= 0 else 'S'}_{abs(izquierda):03d}"
                           f"{'W' if izquierda < 0 else 'E'}")
    return nombres


def descargar(url, destino, minimo_bytes=1 << 20, intentos=3):
    """Baja un archivo por partes, con reintentos. Si ya está, no hace nada."""
    if destino.exists() and destino.stat().st_size > minimo_bytes:
        return destino
    temporal = destino.with_suffix(destino.suffix + ".parcial")
    for intento in range(1, intentos + 1):
        try:
            with requests.get(url, stream=True, timeout=300) as r:
                r.raise_for_status()
                with temporal.open("wb") as f:
                    for trozo in r.iter_content(1 << 20):
                        f.write(trozo)
            if temporal.stat().st_size <= minimo_bytes:
                raise RuntimeError("el archivo llegó demasiado pequeño")
            temporal.replace(destino)
            print(f"  listo: {destino.name} ({destino.stat().st_size / 1e6:.0f} MB)")
            return destino
        except Exception as e:
            print(f"  {destino.name}: intento {intento} de {intentos} falló ({str(e)[:120]})")
            temporal.unlink(missing_ok=True)
    raise RuntimeError(f"No se pudo bajar {url}. Vuelve a correr esta celda.")


for cuadro in cuadros_hansen(BBOX):
    for capa in ("treecover2000", "lossyear"):
        nombre = f"Hansen_{HANSEN_VERSION}_{capa}_{cuadro}.tif"
        descargar(f"https://storage.googleapis.com/earthenginepartners-hansen/{HANSEN_VERSION}/{nombre}",
                  DIR_HANSEN / nombre)
descargar("https://bart.ideam.gov.co/smbyc/Cambio%20en%20la%20superficie%20cubierta%20por%20bosque%20natural/"
          f"Capas/{CAPA_IDEAM_2021_2022}", DIR_TRABAJO / "cambio_2021-2022.img")


def reproyectar_sobre_bloque(rutas, perfil, dtype=np.uint8):
    """Reproyecta uno o varios rásteres sobre la rejilla y los combina (0 = sin dato)."""
    salida = np.zeros((perfil["height"], perfil["width"]), dtype=dtype)
    for ruta in rutas:
        with rasterio.open(ruta) as src:
            trozo = np.zeros_like(salida)
            reproject(source=rasterio.band(src, 1), destination=trozo,
                      dst_transform=perfil["transform"], dst_crs=perfil["crs"],
                      resampling=Resampling.nearest, src_nodata=None, dst_nodata=0)
        salida = np.maximum(salida, trozo)
    return salida

## 6. La rejilla de trabajo

Las fuentes llegan en proyecciones distintas: Landsat en UTM, Hansen y GLAD-L en grados. En vez de adoptar
la rejilla de alguna, se define **una propia** de 30 m en UTM 18N y todas se reproyectan encima, igual que
hace el panel nacional con sus bloques. Así un píxel significa lo mismo en todas las capas. La zona se parte
en dos por la mitad: **el oeste calibra y el este valida**.

In [ ]:
def perfil_zona() -> dict:
    """
    Raster sintetico que cubre la zona piloto, en CRS a ESCALA_M.

    Es el mismo patron que usa zonal.perfil_bloque() para el panel
    nacional: en vez de adoptar la rejilla de alguna de las fuentes, se
    define una propia y TODAS se reproyectan sobre ella. Asi ninguna
    fuente impone su alineacion a las demas, y un pixel significa lo
    mismo en las cuatro capas.

    Los limites se redondean a multiplos de ESCALA_M para que la rejilla
    quede alineada y no aparezcan medios pixeles en los bordes.
    """
    izq, abajo, der, arriba = transform_bounds("EPSG:4326", CRS, *BBOX, densify_pts=21)
    izq, abajo = np.floor(izq / ESCALA_M) * ESCALA_M, np.floor(abajo / ESCALA_M) * ESCALA_M
    der, arriba = np.ceil(der / ESCALA_M) * ESCALA_M, np.ceil(arriba / ESCALA_M) * ESCALA_M

    ancho = int(round((der - izq) / ESCALA_M))
    alto = int(round((arriba - abajo) / ESCALA_M))
    # Affine(a, b, c, d, e, f): x = a*col + c ; y = e*fila + f.
    # e negativo porque las filas avanzan hacia abajo y la Y geografica
    # hacia arriba (convencion "north-up").
    return {"transform": Affine(ESCALA_M, 0, izq, 0, -ESCALA_M, arriba),
            "crs": CRS, "height": alto, "width": ancho,
            "bounds": (izq, abajo, der, arriba)}


def mascara_mitades(perfil: dict) -> Tuple[np.ndarray, np.ndarray]:
    """
    Divide la zona por su meridiano central: oeste calibra, este valida.

    El corte se hace sobre la rejilla proyectada, no sobre longitudes,
    para que ambas mitades tengan exactamente el mismo numero de
    columnas y ninguna quede sistematicamente mas grande.
    """
    mitad = perfil["width"] // 2
    oeste = np.zeros((perfil["height"], perfil["width"]), dtype=bool)
    oeste[:, :mitad] = True
    return oeste, ~oeste

## 7. Landsat: buscar las escenas, leer solo el recorte y armar el compuesto

- **Buscar.** Una consulta al catálogo STAC de Planetary Computer trae las escenas Landsat 8 y 9 sobre la zona.
  No se filtran por nubosidad de escena: una escena muy nublada puede tener despejada justo la esquina que
  interesa. El filtro es por píxel, con la banda de calidad.
- **Leer solo el recorte.** Los archivos son COG (Cloud Optimized GeoTIFF): con `/vsicurl/`, GDAL lee por
  internet solo los bloques que cubren la zona, megabytes en vez del gigabyte que pesa cada escena.
- **De reflectancia a NBR.** Se descartan los píxeles con nube, borde de nube o **sombra de nube** (la sombra
  simula pérdida de biomasa), se aplica la ganancia y el desplazamiento de Collection 2 y se calcula
  `NBR = (NIR - SWIR2) / (NIR + SWIR2)`.
- **El compuesto de mediana.** Para cada píxel, la mediana de todas sus lecturas limpias de la ventana, que
  resiste mejor que la media a una lectura contaminada. Se guarda también cuántas lecturas limpias tuvo y su
  fecha mediana. Se trabaja por franjas de filas para no llenar la memoria.
- **El caché.** Componer las dos ventanas tarda varios minutos y no depende de los umbrales, así que se guarda
  en la carpeta de trabajo con una clave que incluye zona, fechas y rejilla.

In [ ]:
def buscar_escenas(inicio: str, fin: str) -> List[dict]:
    """
    Consulta el catalogo STAC de Planetary Computer.

    Se usa requests directamente en vez de pystac-client para no sumar
    dependencias: la consulta es un unico POST con un JSON, y el
    pipeline ya trae requests.

    NO se filtra por nubosidad de escena. Una escena con 80 % de nubes
    puede tener perfectamente despejada la esquina que nos interesa, y
    descartarla reduciria observaciones justo donde mas falta hacen. El
    filtro real es por pixel, con QA_PIXEL.
    """
    q = {"collections": [COLECCION_STAC], "bbox": list(BBOX),
         "datetime": f"{inicio}/{fin}",
         "query": {"platform": {"in": PLATAFORMAS}}, "limit": 100}
    try:
        r = requests.post(STAC_URL, json=q, timeout=TIEMPO_ESPERA)
        r.raise_for_status()
        escenas = r.json().get("features", [])
    except Exception as e:
        raise RuntimeError(f"No se pudo consultar el catalogo STAC:\n  {e}")

    if not escenas:
        raise RuntimeError(
            f"El catalogo no devolvio escenas Landsat 8/9 entre {inicio} y {fin} "
            f"sobre la zona piloto. Revise BBOX y las fechas.")

    por_plat: Dict[str, int] = {}
    for e in escenas:
        p = e["properties"].get("platform", "?")
        por_plat[p] = por_plat.get(p, 0) + 1
    logger.info("  %s a %s: %d escenas  (%s)", inicio, fin, len(escenas),
                ", ".join(f"{k}: {v}" for k, v in sorted(por_plat.items())))
    return escenas


def token_sas() -> str:
    """
    Token de lectura anonimo para el almacenamiento de Planetary
    Computer. No exige cuenta: lo entrega el servicio a quien lo pida.
    """
    try:
        r = requests.get(SAS_URL, timeout=TIEMPO_ESPERA)
        r.raise_for_status()
        return r.json()["token"]
    except Exception as e:
        raise RuntimeError(f"No se pudo obtener el token de lectura:\n  {e}")


def _leer_ventana(url: str, perfil: dict, fila0: int, n_filas: int,
                  dtype) -> Optional[np.ndarray]:
    """
    Lee de un COG remoto SOLO el rectangulo que cae sobre una franja de
    la rejilla de trabajo, y lo reproyecta sobre ella.

    Aqui esta el ahorro que hace viable trabajar sin Earth Engine: una
    escena Landsat completa pesa del orden de 1 GB, pero al ser COG el
    servidor entrega unicamente los bloques que cubren la ventana
    pedida. Se descargan megabytes, no gigabytes.

    Devuelve None si la escena no intersecta la franja.
    """
    tr = perfil["transform"]
    arriba = tr.f + tr.e * fila0
    abajo = tr.f + tr.e * (fila0 + n_filas)
    izq, der = tr.c, tr.c + tr.a * perfil["width"]

    destino = np.zeros((n_filas, perfil["width"]), dtype=dtype)
    tr_franja = Affine(tr.a, 0, izq, 0, tr.e, arriba)

    try:
        with rasterio.open(f"/vsicurl/{url}") as src:
            # Limites de la franja expresados en el CRS de la escena.
            b = transform_bounds(perfil["crs"], src.crs, izq, abajo, der, arriba,
                                 densify_pts=21)
            sb = src.bounds
            if b[2] <= sb.left or b[0] >= sb.right or b[3] <= sb.bottom or b[1] >= sb.top:
                return None                      # sin traslape

            ven = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
            # Un margen de un pixel evita perder la fila o columna del
            # borde al remuestrear.
            ven = Window(max(0, ven.col_off - 1), max(0, ven.row_off - 1),
                         min(ven.width + 2, src.width), min(ven.height + 2, src.height))
            if ven.width <= 0 or ven.height <= 0:
                return None

            datos = src.read(1, window=ven)
            reproject(source=datos, destination=destino,
                      src_transform=src.window_transform(ven), src_crs=src.crs,
                      dst_transform=tr_franja, dst_crs=perfil["crs"],
                      resampling=Resampling.nearest,
                      src_nodata=0, dst_nodata=0)
    except Exception as e:
        logger.warning("      no se pudo leer una escena (%s); se omite", e)
        return None
    return destino


def _nbr_de_escena(escena: dict, sas: str, perfil: dict,
                   fila0: int, n_filas: int) -> Optional[np.ndarray]:
    """
    NBR de una escena sobre una franja, con nubes enmascaradas.

    Bits de QA_PIXEL que se descartan:
        1  nube dilatada  (borde difuso alrededor de la nube)
        3  nube
        4  sombra de nube (tan daniana como la nube: baja la
                           reflectancia y simula perdida de biomasa)

    Los valores de Collection 2 nivel 2 vienen como enteros escalados;
    hay que aplicar la ganancia 0.0000275 y el desplazamiento -0.2 para
    volver a reflectancia fisica. Sin esa conversion el NBR sigue siendo
    un cociente normalizado y "parece" correcto, pero no lo es.

    Devuelve un array float32 con NaN donde no hay observacion valida.
    """
    a = escena["assets"]
    nir = _leer_ventana(f"{a['nir08']['href']}?{sas}", perfil, fila0, n_filas, np.uint16)
    if nir is None:
        return None
    swir = _leer_ventana(f"{a['swir22']['href']}?{sas}", perfil, fila0, n_filas, np.uint16)
    qa = _leer_ventana(f"{a['qa_pixel']['href']}?{sas}", perfil, fila0, n_filas, np.uint16)
    if swir is None or qa is None:
        return None

    limpio = ((qa & (1 << 1)) == 0) & ((qa & (1 << 3)) == 0) & ((qa & (1 << 4)) == 0)
    valido = limpio & (nir > 0) & (swir > 0) & (qa > 0)

    nir_r = nir.astype(np.float32) * 0.0000275 - 0.2
    swir_r = swir.astype(np.float32) * 0.0000275 - 0.2
    suma = nir_r + swir_r
    with np.errstate(divide="ignore", invalid="ignore"):
        nbr = (nir_r - swir_r) / suma
    nbr[~valido | (suma == 0)] = np.nan
    return nbr.astype(np.float32)


def _fecha_escena(escena: dict) -> dt.date:
    """Fecha de adquisicion de una escena, tomada de su registro STAC."""
    return dt.date.fromisoformat(escena["properties"]["datetime"][:10])


def compuesto_nbr(escenas: List[dict], sas: str, perfil: dict,
                  referencia: dt.date, etiqueta: str
                  ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compuesto de MEDIANA del NBR, conteo de observaciones limpias y fecha
    efectiva de cada pixel.

    Se usa mediana y no media porque es robusta a las observaciones
    residuales que la mascara de nubes deja pasar: una sola lectura
    contaminada desplaza la media, pero no la mediana.

    TODAS las lecturas limpias de la ventana pesan igual, sin importar su
    fecha. Se probo privilegiar las cercanas al centro de la ventana --
    mediana ponderada con peso gaussiano de 15 y de 30 dias, las 3
    lecturas mas cercanas, y solo la mas cercana-- y todas empeoraron la
    concordancia con GLAD-L: el F1 bajo de 0,505 a entre 0,455 y 0,497.
    El error dominante de este metodo es la nube residual, no la fecha;
    usar menos lecturas gana precision de fecha y pierde la proteccion
    que da la mediana.

    La FECHA EFECTIVA se devuelve como desfase en dias respecto de
    `referencia` (el centro de la ventana): la mediana de las fechas de
    las lecturas limpias de cada pixel. Dice que momento representa en
    realidad el compuesto, y con eso que intervalo mide el dNBR. Ver
    fecha_efectiva().

    El trabajo va por FRANJAS horizontales para acotar la memoria.
    Apilar todas las escenas de la zona completa exigiria del orden de
    varios cientos de MB simultaneos; por franjas, el pico queda en
    decenas y el resultado es identico.
    """
    alto, ancho = perfil["height"], perfil["width"]
    nbr = np.full((alto, ancho), np.nan, dtype=np.float32)
    obs = np.zeros((alto, ancho), dtype=np.uint16)
    desfase = np.full((alto, ancho), np.nan, dtype=np.float32)

    for fila0 in range(0, alto, FILAS_POR_FRANJA):
        n = min(FILAS_POR_FRANJA, alto - fila0)
        pila, dias = [], []
        for e in escenas:
            c = _nbr_de_escena(e, sas, perfil, fila0, n)
            if c is not None:
                pila.append(c)
                dias.append((_fecha_escena(e) - referencia).days)
        if not pila:
            logger.warning("    %s franja %d-%d: sin observaciones", etiqueta,
                           fila0, fila0 + n)
            continue
        cubo = np.stack(pila)
        limpio = ~np.isnan(cubo)
        fechas = np.where(limpio, np.array(dias, np.float32)[:, None, None],
                          np.nan)
        with warnings.catch_warnings():
            # nanmedian avisa "All-NaN slice" cuando un pixel estuvo
            # nublado en TODAS las escenas. Es el caso esperado y no un
            # fallo: ese pixel queda en NaN, no entra al dominio, y
            # termina contado en el diagnostico de nubes. Se silencia el
            # aviso para que no tape el registro util de la corrida.
            warnings.simplefilter("ignore", RuntimeWarning)
            nbr[fila0:fila0 + n] = np.nanmedian(cubo, axis=0)
            desfase[fila0:fila0 + n] = np.nanmedian(fechas, axis=0)
        obs[fila0:fila0 + n] = limpio.sum(axis=0).astype(np.uint16)
        logger.info("    %s franja %4d-%4d de %d  (%d escenas con datos)",
                    etiqueta, fila0, fila0 + n, alto, len(pila))
    return nbr, obs, desfase


def compuestos_con_cache(perfil: dict):
    """
    Compuestos de T1 y T2, reutilizando el resultado si ya se calculo.

    Componer las dos ventanas exige leer del orden de 70 escenas por
    HTTP y toma varios minutos. Ese resultado depende UNICAMENTE de la
    zona, las fechas y la rejilla, no de los umbrales de deteccion, asi
    que se guarda en disco: ajustar UMBRALES_DNBR o PIXELES_MIN_PARCHE y
    volver a correr pasa a costar segundos.

    La clave del archivo incluye los parametros que SI cambian el
    contenido -- zona, fechas y forma de la rejilla-- para que un cambio
    en cualquiera de ellos invalide el cache en vez de devolver
    calladamente un compuesto de otra corrida. Es el mismo cuidado que
    calcular_bosque.py tiene con su propio cache.

    Un cache de una version anterior del modulo, sin la fecha efectiva,
    se descarta y se recalcula.
    """
    clave = (f"nbr_{BBOX[0]}_{BBOX[1]}_{BBOX[2]}_{BBOX[3]}"
             f"_{T1_INICIO}_{T1_FIN}_{T2_INICIO}_{T2_FIN}"
             f"_{perfil['width']}x{perfil['height']}").replace("-", "")
    archivo = DIR_DEMO / f"cache_{clave}.npz"
    campos = ("nbr_t1", "obs_t1", "desfase_t1", "nbr_t2", "obs_t2", "desfase_t2")

    if archivo.exists():
        d = np.load(archivo)
        if all(c in d.files for c in campos):
            logger.info("  reutilizando compuestos en cache (%s)", archivo.name)
            return tuple(d[c] for c in campos)
        logger.info("  el cache no trae la fecha efectiva (version anterior); "
                    "se recalcula")

    e_t1 = buscar_escenas(T1_INICIO, T1_FIN)
    e_t2 = buscar_escenas(T2_INICIO, T2_FIN)
    sas = token_sas()
    logger.info("  componiendo T1...")
    nbr_t1, obs_t1, des_t1 = compuesto_nbr(e_t1, sas, perfil, GLAD_DESDE, "T1")
    logger.info("  componiendo T2...")
    nbr_t2, obs_t2, des_t2 = compuesto_nbr(e_t2, sas, perfil, GLAD_HASTA, "T2")

    np.savez_compressed(archivo, nbr_t1=nbr_t1, obs_t1=obs_t1, desfase_t1=des_t1,
                        nbr_t2=nbr_t2, obs_t2=obs_t2, desfase_t2=des_t2)
    logger.info("  compuestos guardados en %s", archivo.name)
    return nbr_t1, obs_t1, des_t1, nbr_t2, obs_t2, des_t2

## 8. El bosque inicial

Un píxel es bosque si en 2000 tenía al menos 30 % de dosel **y** no se había perdido antes de 2022. Lo segundo
evita contar como hallazgo un claro que existe desde hace años.

In [ ]:
def mascara_bosque(perfil: dict) -> np.ndarray:
    """
    Bosque existente al inicio de T1, segun Hansen GFC.

    Tres condiciones:

      dosel >= UMBRAL_DOSEL  cobertura arborea suficiente en 2000. Es el
                             mismo umbral que usa el panel nacional
                             (config_local.umbral_dosel), para que la
                             demo hable de "bosque" en los mismos
                             terminos.
      sin perdida previa     si ya se habia deforestado antes de la
                             ventana, no puede volver a perderse dentro
                             de ella. Sin esto la demo contaria como
                             hallazgo un claro que existe hace anios.
      datamask == 1          tierra firme, OPCIONAL. El pipeline solo
                             descarga treecover2000 y lossyear, asi que
                             esta capa suele no estar en disco. Su papel
                             es descartar agua permanente, y el umbral
                             de dosel ya lo hace: un cuerpo de agua
                             tiene 0 % de cobertura arborea en 2000 y no
                             pasa el primer filtro. Se aplica si los
                             granulos estan, y si no, se avisa y se
                             sigue.

    Se reutiliza zonal.reproyectar_sobre_bloque(), la misma funcion que
    usa el panel nacional: la demo no reimplementa la geometria del
    reparto, la hereda.
    """
    def granulos(capa: str, obligatoria: bool = True) -> List[Path]:
        g = sorted(DIR_HANSEN.glob(f"Hansen_*_{capa}_*.tif"))
        if not g and obligatoria:
            raise RuntimeError(
                f"No hay granulos de {capa} en {DIR_HANSEN}. Corre la sección 5 (insumos).")
        return g

    dosel = reproyectar_sobre_bloque(granulos("treecover2000"), perfil)
    perdida = reproyectar_sobre_bloque(granulos("lossyear"), perfil)
    previa = (perdida >= 1) & (perdida <= ANIO_CORTE_PERDIDA)
    bosque = (dosel >= UMBRAL_DOSEL) & ~previa

    g_dm = granulos("datamask", obligatoria=False)
    if g_dm:
        bosque &= reproyectar_sobre_bloque(g_dm, perfil) == 1
        logger.info("  datamask aplicado")
    else:
        logger.info("  datamask no esta en disco; se omite. El umbral de dosel "
                    "ya excluye el agua permanente (0 %% de cobertura en 2000).")
    return bosque

## 9. GLAD-L

El almacenamiento de GFW cobra la descarga a quien la pide ("Requester Pays"), así que se pasa por la API con
la llave, que responde con una dirección ya firmada. La banda `date_conf` empaqueta confianza y fecha en un
entero (`confianza × 10.000 + días desde 2014-12-31`); la codificación se comprueba contra los datos en vez de
darla por supuesta, porque si estuviera mal el resultado saldría vacío sin error. Si la API responde a medias
(pasa mientras GFW publica una versión nueva), se reintenta; si rechaza la llave, se detiene de una.

In [ ]:
def _api_key() -> str:
    """La API key de GFW de la celda 1."""
    return GFW_API_KEY.strip()


def _version_glad(clave: str, intentos: int = 3, espera_s: int = 10) -> str:
    """
    Version vigente de GLAD-L, reintentando ante respuestas incompletas.

    El endpoint /latest puede responder 200 con un cuerpo sin el campo
    "data" mientras GFW publica una version nueva del dataset. Dura poco
    y se resuelve solo, pero antes reventaba con un KeyError sin
    explicacion. Aqui se separa ese caso de los que NO se arreglan
    reintentando -- una key invalida, por ejemplo -- y solo se reintenta
    cuando tiene sentido hacerlo.
    """
    u = f"{GFW_API}/dataset/{GLAD_DATASET}/latest"
    motivo = "sin intentos"
    for intento in range(1, intentos + 1):
        try:
            r = requests.get(u, headers={"x-api-key": clave},
                             timeout=TIEMPO_ESPERA)
        except requests.RequestException as e:
            motivo = f"no hubo respuesta del servidor ({e})"
        else:
            if r.status_code in (401, 403):
                raise RuntimeError(
                    f"GFW rechazo la API key (HTTP {r.status_code}).\n"
                    "  Saca una nueva con el anexo del notebook 03 y ponla en GFW_API_KEY (celda 1).")
            if r.status_code != 200:
                motivo = f"HTTP {r.status_code}"
            else:
                try:
                    return r.json()["data"]["version"]
                except (ValueError, KeyError, TypeError):
                    motivo = ("respuesta 200 sin el campo data.version; "
                              "GFW suele estar publicando una version nueva")

        if intento < intentos:
            logger.warning("  no se pudo resolver la version (%s); "
                           "reintento %d de %d en %d s",
                           motivo, intento, intentos - 1, espera_s)
            time.sleep(espera_s)

    raise RuntimeError(
        f"No se pudo resolver la version de {GLAD_DATASET} tras "
        f"{intentos} intentos.\n"
        f"  Ultimo motivo: {motivo}\n"
        "  Si fue una respuesta incompleta, suele bastar con volver a "
        "correr en unos minutos.")


def _url_glad(clave: str) -> str:
    """
    URL firmada del raster date_conf de GLAD-L.

    El bucket de GFW es "Requester Pays": una descarga anonima falla con
    AccessDenied. Hay que pasar por el endpoint de la API con la key,
    que responde 307 hacia una URL de S3 ya firmada.
    """
    v = _version_glad(clave)

    u = (f"{GFW_API}/dataset/{GLAD_DATASET}/{v}/download/geotiff"
         f"?grid=10/100000&tile_id={GLAD_TILE}&pixel_meaning=date_conf")
    r = requests.get(u, headers={"x-api-key": clave},
                     allow_redirects=False, timeout=TIEMPO_ESPERA)
    if r.status_code != 307 or "Location" not in r.headers:
        raise RuntimeError(
            f"El endpoint de descarga de GLAD-L respondio {r.status_code}, "
            f"se esperaba 307 con redireccion.\n  {r.text[:200]}")
    logger.info("  GLAD-L version %s, tesela %s", v, GLAD_TILE)
    return r.headers["Location"]


def leer_glad(perfil: dict, clave: str) -> np.ndarray:
    """
    Raster date_conf de GLAD-L sobre la rejilla de trabajo, sin filtrar.

    CODIFICACION de la banda date_conf: el valor empaqueta dos cosas,
    confianza y fecha, como conf * 10000 + dias desde GLAD_EPOCA. Se
    verifica empiricamente al leer -- se imprimen los codigos de
    confianza y el rango de fechas observados-- en vez de darla por
    supuesta: un desajuste aqui produciria un resultado vacio, que es el
    modo de falla mas dificil de detectar.

    Se lee una sola vez por corrida. alertas_glad() lo filtra para cada
    ventana que haga falta: la de la demo, y el anio calendario de la
    referencia entre productos.
    """
    url = _url_glad(clave)
    os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

    tr = perfil["transform"]
    izq, arriba = tr.c, tr.f
    der = izq + tr.a * perfil["width"]
    abajo = arriba + tr.e * perfil["height"]
    destino = np.zeros((perfil["height"], perfil["width"]), dtype=np.uint16)

    try:
        with rasterio.open(f"/vsicurl/{url}") as src:
            b = transform_bounds(perfil["crs"], src.crs, izq, abajo, der, arriba,
                                 densify_pts=21)
            ven = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
            datos = src.read(1, window=ven)
            reproject(source=datos, destination=destino,
                      src_transform=src.window_transform(ven), src_crs=src.crs,
                      dst_transform=tr, dst_crs=perfil["crs"],
                      resampling=Resampling.nearest, src_nodata=0, dst_nodata=0)
    except Exception as e:
        raise RuntimeError(f"No se pudo leer el raster de GLAD-L:\n  {e}")

    con_dato = destino > 0
    if not con_dato.any():
        raise RuntimeError(
            "El raster de GLAD-L no trae ninguna alerta sobre la zona piloto. "
            "Revise GLAD_TILE y BBOX.")

    conf = (destino // 10000).astype(np.uint8)
    dias = (destino % 10000).astype(np.int32)
    d_min, d_max = int(dias[con_dato].min()), int(dias[con_dato].max())
    logger.info("  codigos de confianza observados: %s",
                sorted(np.unique(conf[con_dato]).tolist()))
    logger.info("  rango de fechas: %s a %s",
                GLAD_EPOCA + dt.timedelta(days=d_min),
                GLAD_EPOCA + dt.timedelta(days=d_max))
    return destino


def alertas_glad(destino: np.ndarray, desde: dt.date = GLAD_DESDE,
                 hasta: dt.date = GLAD_HASTA) -> np.ndarray:
    """
    Alertas GLAD-L confirmadas entre `desde` y `hasta`, ambos incluidos.

    Si la ventana pedida cae fuera del rango de fechas que trae el
    raster, se detiene con un mensaje claro. Devolver cero alertas seria
    peor: pareceria que en la zona no hubo perdida.
    """
    con_dato = destino > 0
    conf = (destino // 10000).astype(np.uint8)
    dias = (destino % 10000).astype(np.int32)
    d_min, d_max = int(dias[con_dato].min()), int(dias[con_dato].max())
    i_desde = (desde - GLAD_EPOCA).days
    i_hasta = (hasta - GLAD_EPOCA).days
    if d_max < i_desde or d_min > i_hasta:
        raise RuntimeError(
            f"Las alertas disponibles ({GLAD_EPOCA + dt.timedelta(days=d_min)} a "
            f"{GLAD_EPOCA + dt.timedelta(days=d_max)}) no cubren la ventana pedida "
            f"({desde} a {hasta}). Ajuste T1/T2 o GLAD_DESDE/GLAD_HASTA.")
    return (con_dato & (conf >= GLAD_CONF_MIN)
            & (dias >= i_desde) & (dias <= i_hasta))

## 10. El dominio

Es la decisión más importante de la demo. Un píxel de bosque tapado por nubes en T1 o en T2 **no se puede
evaluar**: contarlo como "sin pérdida" sería un verdadero negativo regalado que infla la exactitud y esconde
la falta de observación. Por eso se excluye, y lo excluido se reporta. Se reporta también qué fecha
representa en realidad cada compuesto, y con eso qué intervalo mide el dNBR.

In [ ]:
def construir_dominio(bosque, obs_t1, obs_t2) -> np.ndarray:
    """
    Dominio = bosque inicial CON al menos una observacion limpia en cada
    ventana.

    Es la decision metodologica mas importante del modulo. Un pixel de
    bosque tapado por nubes en T1 o en T2 no se puede evaluar: no se
    sabe si cambio. Tratarlo como "sin perdida" lo convertiria en un
    verdadero negativo gratuito, inflando la exactitud global y
    escondiendo el problema real, que es la falta de observacion.

    Al excluirlos, la metrica describe solo el terreno donde de verdad
    hubo con que mirar, y el porcentaje excluido pasa a ser un resultado
    que se reporta.
    """
    return bosque & (obs_t1 >= 1) & (obs_t2 >= 1)


def diagnostico_nubes(bosque, dominio, obs_t1, obs_t2) -> pd.DataFrame:
    """
    Cuanto bosque quedo observable, y con cuantas observaciones limpias.

    Se reportan DOS cosas distintas, y conviene no confundirlas:

      sin_datos_pct   bosque que quedo FUERA del dominio por no tener
                      ninguna observacion limpia en alguna de las dos
                      ventanas. Con tres meses de Landsat 8 y 9 este
                      numero suele ser cero, porque basta UNA observacion
                      para entrar.

      obs_p10 / mediana   cuantas observaciones limpias tuvo de verdad
                      cada pixel. Aqui es donde se ve el problema de
                      nubosidad: que la mediana sea de apenas unas pocas
                      escenas -- de las decenas disponibles-- significa
                      que la mayoria se descarto por nubes, y que el
                      compuesto descansa en muy pocas lecturas.

    Un pixel con una sola observacion limpia entra al dominio, pero su
    mediana es esa unica lectura: si venia contaminada y la mascara no la
    atrapo, el dNBR de ese pixel es ruido. Por eso el percentil 10 dice
    mas sobre la calidad del compuesto que el porcentaje excluido.
    """
    ha = (ESCALA_M ** 2) / 10_000.0
    b, d = int(bosque.sum()), int(dominio.sum())
    o1, o2 = obs_t1[bosque], obs_t2[bosque]
    return pd.DataFrame([{
        "bosque_inicial_ha": round(b * ha, 1),
        "con_obs_t1_ha": round(int((bosque & (obs_t1 >= 1)).sum()) * ha, 1),
        "con_obs_t2_ha": round(int((bosque & (obs_t2 >= 1)).sum()) * ha, 1),
        "dominio_ha": round(d * ha, 1),
        "sin_datos_ha": round((b - d) * ha, 1),
        "sin_datos_pct": round(100 * (b - d) / b, 2) if b else None,
        "obs_min_t1": int(o1.min()) if b else 0,
        "obs_p10_t1": int(np.percentile(o1, 10)) if b else 0,
        "obs_mediana_t1": int(np.median(o1)) if b else 0,
        "obs_max_t1": int(o1.max()) if b else 0,
        "obs_min_t2": int(o2.min()) if b else 0,
        "obs_p10_t2": int(np.percentile(o2, 10)) if b else 0,
        "obs_mediana_t2": int(np.median(o2)) if b else 0,
        "obs_max_t2": int(o2.max()) if b else 0,
        "pct_bosque_con_1_sola_obs": round(
            100 * float(((o1 == 1) | (o2 == 1)).mean()), 2) if b else None,
    }])


def fecha_efectiva(dominio, desfase_t1, desfase_t2) -> pd.DataFrame:
    """
    Que periodo mide en realidad el dNBR.

    Cada compuesto usa todas las lecturas limpias de enero a marzo con el
    mismo peso, asi que no representa necesariamente el centro de la
    ventana: si las lecturas limpias de un pixel cayeron sobre todo en
    enero, su compuesto describe enero. La fecha efectiva de cada pixel
    es la mediana de las fechas de sus lecturas limpias, y el intervalo
    efectivo es la distancia entre la de T1 y la de T2.

    El dato "anual" de la demo es por eso aproximado. Esta tabla dice
    cuanto: una fila por medida, con percentiles sobre el dominio.

        desfase_t1_dias  dias entre la fecha efectiva de T1 y GLAD_DESDE
        desfase_t2_dias  dias entre la fecha efectiva de T2 y GLAD_HASTA
        intervalo_dias   dias entre las dos fechas efectivas

    Las dos primeras filas traen ademas la fecha efectiva mediana, y la
    ultima el porcentaje de pixeles cuyo intervalo se aparta del anio en
    mas de 30 y de 45 dias.
    """
    nominal = (GLAD_HASTA - GLAD_DESDE).days
    d1 = desfase_t1[dominio].astype(np.float64)
    d2 = desfase_t2[dominio].astype(np.float64)
    intervalo = nominal + d2 - d1

    filas = []
    for medida, v, ref in (("desfase_t1_dias", d1, GLAD_DESDE),
                           ("desfase_t2_dias", d2, GLAD_HASTA),
                           ("intervalo_dias", intervalo, None)):
        p5, p25, p50, p75, p95 = np.percentile(v, [5, 25, 50, 75, 95])
        fila = {"medida": medida,
                "p5": round(p5), "p25": round(p25), "mediana": round(p50),
                "p75": round(p75), "p95": round(p95),
                "fecha_mediana": "", "pct_mas_de_30d_del_anio": None,
                "pct_mas_de_45d_del_anio": None}
        if ref is not None:
            fila["fecha_mediana"] = (ref + dt.timedelta(days=round(p50))).isoformat()
        else:
            fila["pct_mas_de_30d_del_anio"] = round(
                100 * float((np.abs(v - nominal) > 30).mean()), 1)
            fila["pct_mas_de_45d_del_anio"] = round(
                100 * float((np.abs(v - nominal) > 45).mean()), 1)
        filas.append(fila)
    return pd.DataFrame(filas)

## 11. Detección y filtro de parche

Un píxel se marca si su dNBR supera el umbral y está en el dominio, y se queda solo si pertenece a un parche de
al menos 12 píxeles conectados (vecindad de 8, para no partir un claro en diagonal). **A GLAD-L se le aplica el
mismo tratamiento**: la misma máscara de bosque, el mismo dominio y el mismo filtro de área mínima. Sin eso la
comparación mediría diferencias de encuadre, no de detección.

In [ ]:
def filtrar_parches(binaria: np.ndarray) -> np.ndarray:
    """
    Conserva solo los pixeles que pertenecen a un parche de al menos
    PIXELES_MIN_PARCHE pixeles conectados, con vecindad de 8.

    La vecindad de 8 (y no de 4) importa: un claro alargado en diagonal
    -- una via de extraccion, por ejemplo-- se partiria en fragmentos
    sueltos con vecindad de 4 y se perderia entero al filtrar.
    """
    estructura = np.ones((3, 3), dtype=bool)          # 8 vecinos
    etiquetas, n = ndimage.label(binaria, structure=estructura)
    if n == 0:
        return np.zeros_like(binaria, dtype=bool)
    tam = np.bincount(etiquetas.ravel())
    tam[0] = 0                                        # el fondo no cuenta
    return np.isin(etiquetas, np.flatnonzero(tam >= PIXELES_MIN_PARCHE))


def perdida_propia(dnbr, dominio, umbral: float) -> np.ndarray:
    """Perdida derivada del dNBR: umbral, dominio y filtro de parche."""
    with np.errstate(invalid="ignore"):
        cruda = (dnbr > umbral) & dominio
    return filtrar_parches(cruda)


def perdida_glad(glad, dominio) -> np.ndarray:
    """
    Perdida segun GLAD-L, con EXACTAMENTE el mismo tratamiento que la
    propia: misma mascara de bosque y mismo dominio (ambos vienen dentro
    de 'dominio'), y el mismo filtro de area minima. El filtro de fecha
    ya se aplico al leer el raster.

    Sin esas igualaciones la comparacion mediria diferencias de encuadre.
    Un ejemplo concreto: GLAD-L alerta tambien fuera del bosque definido
    por Hansen, de modo que sin la mascara comun apareceria como
    "deteccion adicional" algo que la propia nunca pudo haber encontrado.
    """
    return filtrar_parches(glad & dominio)

## 12. Las métricas y la calibración

Precisión, sensibilidad y F1 tomando GLAD-L como **referencia**, que no es lo mismo que "verdad": miden
concordancia con un producto publicado. La exactitud global se reporta con el nombre
`exactitud_global_no_informativa` porque el bosque estable domina la escena y un detector que no marcara nada
sacaría una cifra parecida. El umbral se calibra **solo en la mitad oeste** y se evalúa en la este, que la
calibración no vio.

In [ ]:
def matriz_acuerdo(propia, glad, dominio, region) -> dict:
    """Hectareas en las cuatro combinaciones propia/GLAD dentro del dominio."""
    ha = (ESCALA_M ** 2) / 10_000.0
    d = dominio & region
    return {
        "ambos_ha": round(int((d & propia & glad).sum()) * ha, 1),
        "solo_propia_ha": round(int((d & propia & ~glad).sum()) * ha, 1),
        "solo_glad_ha": round(int((d & ~propia & glad).sum()) * ha, 1),
        "ninguno_ha": round(int((d & ~propia & ~glad).sum()) * ha, 1),
    }


def metricas(m: dict) -> dict:
    """
    Precision, sensibilidad y F1 tomando GLAD-L como referencia.

    ADVERTENCIA sobre la exactitud global: se incluye porque suele
    pedirse, pero NO es informativa aqui. El bosque estable domina la
    escena -- suele ser mas del 95 % del dominio--, asi que un detector
    que no marcara nada obtendria una exactitud altisima. Las metricas
    que si dicen algo son las tres primeras.

    "Referencia" tampoco significa "verdad": GLAD-L tiene sus propios
    errores. Lo que miden estas cifras es CONCORDANCIA con un producto
    publicado, no exactitud contra el terreno. Para eso esta la muestra
    de validacion visual.
    """
    vp, fp, fn, vn = (m["ambos_ha"], m["solo_propia_ha"],
                      m["solo_glad_ha"], m["ninguno_ha"])
    prec = vp / (vp + fp) if (vp + fp) else 0.0
    sens = vp / (vp + fn) if (vp + fn) else 0.0
    f1 = 2 * prec * sens / (prec + sens) if (prec + sens) else 0.0
    tot = vp + fp + fn + vn
    return {"precision": round(prec, 4), "sensibilidad": round(sens, 4),
            "f1": round(f1, 4),
            "exactitud_global_no_informativa": round((vp + vn) / tot, 4) if tot else 0.0}


def calibrar_umbral(dnbr, dominio, glad_p, oeste) -> pd.DataFrame:
    """
    Recorre los umbrales candidatos en la MITAD OESTE.

    La mitad oeste no vuelve a tocarse despues: el umbral elegido aqui
    se aplica tal cual sobre la mitad este, que el procedimiento no ha
    visto.
    """
    filas = []
    for u in UMBRALES_DNBR:
        p = perdida_propia(dnbr, dominio, u)
        m = matriz_acuerdo(p, glad_p, dominio, oeste)
        fila = {"umbral_dnbr": u, **m, **metricas(m)}
        filas.append(fila)
        logger.info("  umbral %.2f -> precision %.3f | sensibilidad %.3f | F1 %.3f",
                    u, fila["precision"], fila["sensibilidad"], fila["f1"])
    return pd.DataFrame(filas)

## 13. De píxeles a celdas de 5 km, y la referencia entre productos

- **Detección por claro**: cuántos claros de una capa toca la otra, sin exigir que coincida el contorno.
- **Curva de observabilidad**: la concordancia según cuántas lecturas limpias se exijan. Se filtra por
  observabilidad, nunca por resultado.
- **Comparación por celda de 5 km**, la escala de los modelos: dos productos pueden discrepar píxel a píxel y
  coincidir en cuánta pérdida hay en cada celda. Se mide con el Pearson y con la **concordancia de Lin**, que
  además castiga que una fuente reporte más que la otra.
- **Referencia entre productos**: cuánto concuerdan entre sí GLAD-L, Hansen y el IDEAM con la misma vara, para
  poder leer el F1 de la demo.

In [ ]:
def deteccion_por_parche(propia, glad, region) -> pd.DataFrame:
    """
    Cuantos claros se detectan, en vez de cuantos pixeles se aciertan.

    El F1 por pixel castiga igual dos errores muy distintos: inventarse
    un claro donde no lo hay, y encontrar el claro correcto con el
    contorno un poco ancho. Para un panel que tamiza municipios, lo
    segundo importa poco: lo que se necesita saber es si el evento
    quedo senalado.

    Esta funcion agrupa cada capa en claros (componentes conexas, misma
    vecindad de 8 que filtrar_parches) y cuenta cuantos de una capa son
    tocados por al menos un pixel de la otra. Un claro cuenta como
    detectado si hay cualquier traslape, sin exigir que coincida el
    contorno.

    Se reportan las dos direcciones porque responden preguntas
    distintas:
      glad_vs_propia -- de los claros que reporta GLAD-L, cuantos
                        encuentro (omision).
      propia_vs_glad -- de los claros que reporto yo, cuantos confirma
                        GLAD-L (comision).

    Y se reportan dos totales, que no son intercambiables: el conteo
    simple trata igual un claro de 1 ha y uno de 100, mientras que el
    ponderado por area dice cuanta hectarea queda cubierta. En esta zona
    los claros pequenos son la mayoria en numero y poca area, asi que el
    conteo simple sale mucho mas bajo que el ponderado.
    """
    est = np.ones((3, 3), dtype=bool)
    ha = (ESCALA_M ** 2) / 10_000.0
    tramos = [(12, 33, "1-3 ha"), (33, 111, "3-10 ha"),
              (111, 333, "10-30 ha"), (333, 10 ** 9, "mas de 30 ha")]

    filas = []
    for nombre, ref, otro in (("glad_vs_propia", glad, propia),
                              ("propia_vs_glad", propia, glad)):
        lab, n = ndimage.label(ref & region, structure=est)
        if n == 0:
            continue
        tam = np.bincount(lab.ravel())
        tam[0] = 0
        tocado = np.bincount(lab[otro & region].ravel(), minlength=n + 1)
        tocado[0] = 0

        for lo, hi, etiqueta in tramos + [(1, 10 ** 9, "TODOS")]:
            ids = np.flatnonzero((tam >= lo) & (tam < hi))
            if not len(ids):
                continue
            det = ids[tocado[ids] > 0]
            area = float(tam[ids].sum()) * ha
            area_det = float(tam[det].sum()) * ha
            filas.append({
                "direccion": nombre,
                "tamano": etiqueta,
                "parches": len(ids),
                "parches_detectados": len(det),
                "pct_parches": round(100 * len(det) / len(ids), 1),
                "area_ha": round(area, 1),
                "area_detectada_ha": round(area_det, 1),
                "pct_area": round(100 * area_det / area, 1) if area else 0.0,
            })
    return pd.DataFrame(filas)


def curva_observabilidad(propia_fn, glad_raw, bosque, obs_t1, obs_t2,
                         region, umbral: float,
                         minimos=(1, 2, 3, 4, 6, 8, 10)) -> pd.DataFrame:
    """
    Concordancia contra GLAD-L segun cuantas observaciones limpias exija
    el dominio.

    El dominio de la demo pide al menos UNA observacion limpia en cada
    ventana. Subir esa barra deja fuera los pixeles peor observados, y
    la concordancia mejora: con 3 observaciones el F1 ronda 0,29 y con
    10 o mas llega a 0,69. Reportar un solo numero esconde eso.

    La curva se reporta entera porque el valor util depende de para que
    se use: quien necesite cobertura completa lee la primera fila, quien
    pueda restringirse a zonas bien observadas lee las de abajo.

    Se filtra por OBSERVABILIDAD (cuantas veces se pudo ver el pixel),
    nunca por resultado. Filtrar por resultado seria elegir los aciertos.

    Cuidado al leer las filas exigentes: el numero de observaciones lo
    manda la geometria orbital, y en esta zona el traslape entre orbitas
    cae en el sur. Pedir 10 observaciones equivale a quedarse con el
    tercio sur, asi que esa fila describe una region, no la zona. La
    columna cobertura_pct dice cuanta area queda, y reparto_ns cuan
    repartida esta entre el norte y el sur.
    """
    nobs = np.minimum(obs_t1, obs_t2)
    alto = bosque.shape[0]
    norte = np.zeros_like(bosque)
    norte[:alto // 2] = True
    ha = (ESCALA_M ** 2) / 10_000.0
    base = float((bosque & (nobs >= 1) & region).sum()) * ha

    # Dos lecturas de lo mismo, y no son intercambiables:
    #   acumulado -- "exige al menos k": es lo que pasaria si el dominio
    #                subiera su barra, con la cobertura de area que queda.
    #   exacto    -- "solo los pixeles con k observaciones": aisla el
    #                efecto, porque no mezcla los bien observados con los
    #                mal observados. Es la lectura que muestra de verdad
    #                cuanto pesa la nubosidad.
    #
    # Las filas exactas de N bajo hay que leerlas con cuidado: recortar el
    # dominio a esos pixeles los deja dispersos, y filtrar_parches() pide
    # 12 pixeles conexos DENTRO del dominio. Con N=1 no se forma ningun
    # parche y el F1 sale 0,000 por construccion, no porque el metodo
    # falle ahi. El efecto se diluye a medida que el tramo cubre mas area.
    tramos = [("acumulado", k, nobs >= k, str(k)) for k in minimos]
    tramos += [("exacto", lo, (nobs >= lo) & (nobs <= hi), et)
               for lo, hi, et in ((1, 1, "1"), (2, 2, "2"), (3, 3, "3"),
                                  (4, 5, "4-5"), (6, 9, "6-9"), (10, 999, "10+"))]

    filas = []
    for modo, k, cond, etiqueta in tramos:
        dom = bosque & cond
        sel = dom & region
        area = float(sel.sum()) * ha
        if area == 0:
            continue
        gp = perdida_glad(glad_raw, dom)
        pp = propia_fn(dom, umbral)
        m = matriz_acuerdo(pp, gp, dom, region)
        en_norte = float((sel & norte).sum()) * ha
        filas.append({
            "modo": modo,
            "obs": etiqueta,
            "obs_minimas": k,
            "dominio_ha": round(area, 1),
            "cobertura_pct": round(100 * area / base, 1) if base else 0.0,
            "reparto_ns": round(100 * en_norte / area, 1),
            **m, **metricas(m),
        })
    return pd.DataFrame(filas)


def concordancia_lin(referencia, prueba) -> float:
    """
    Coeficiente de concordancia de Lin (1989) entre dos series de
    hectareas por celda.

    Se lee como un Pearson, de -1 a 1, pero mide otra cosa. Pearson dice
    si las dos series SUBEN Y BAJAN JUNTAS, y no le importa si una
    reporta el doble que la otra. Lin exige ademas que los VALORES sean
    iguales: castiga la diferencia de nivel y la de escala.

        Pearson = cov / (sd_ref * sd_pru)
        Lin     = 2 cov / (var_ref + var_pru + (media_ref - media_pru)^2)

    Lin nunca supera en valor absoluto a Pearson, y coincide con el solo
    si las dos series tienen la misma media y la misma dispersion. Con
    las celdas de la demo: Pearson 0,909 y Lin 0,902; si la demo
    reportara el doble en cada celda, Pearson seguiria en 0,909 y Lin
    caeria a 0,668.

    Es la medida que conviene poner como titular, porque responde la
    pregunta del panel: si una fuente dice lo mismo que la otra, celda
    por celda, en donde y en cuanto.

    Lin, L. I-K. (1989). A concordance correlation coefficient to
    evaluate reproducibility. Biometrics, 45(1), 255-268.
    """
    x = np.asarray(referencia, dtype=np.float64)
    y = np.asarray(prueba, dtype=np.float64)
    cov = float(((x - x.mean()) * (y - y.mean())).mean())
    den = float(x.var() + y.var() + (x.mean() - y.mean()) ** 2)
    return 2 * cov / den if den else float("nan")


def comparar_celdas(propia, glad, dominio, region, perfil) -> pd.DataFrame:
    """
    Hectareas perdidas por celda de 5 x 5 km, segun cada fuente.

    Es la comparacion mas relevante para este proyecto: el modelo
    predictivo trabaja a esa escala, no a nivel de pixel. Dos productos
    pueden discrepar pixel a pixel por medio pixel de desplazamiento y
    aun asi coincidir muy bien en cuanta perdida hay en cada celda, que
    es lo que el modelo necesita.

    El indice de celda sale por aritmetica directa sobre la rejilla
    proyectada -- floor(coord / LADO_CELDA_M)-- igual que en zonal.py.
    """
    tr = perfil["transform"]
    px_celda = LADO_CELDA_M // ESCALA_M
    filas = np.arange(perfil["height"])[:, None] // px_celda
    cols = np.arange(perfil["width"])[None, :] // px_celda
    idx = (filas * (perfil["width"] // px_celda + 1) + cols).astype(np.int64)

    ha = (ESCALA_M ** 2) / 10_000.0
    d = dominio & region
    n = idx.max() + 1
    cuenta = lambda m: np.bincount(idx[m], minlength=n) * ha    # noqa: E731

    df = pd.DataFrame({
        "celda": np.arange(n),
        "propia_ha": cuenta(d & propia),
        "glad_ha": cuenta(d & glad),
        "dominio_ha": cuenta(d),
    })
    # Centro de cada celda, en la rejilla proyectada y en lon/lat.
    f_c = (df.celda // (perfil["width"] // px_celda + 1)) * px_celda + px_celda / 2
    c_c = (df.celda % (perfil["width"] // px_celda + 1)) * px_celda + px_celda / 2
    x = tr.c + tr.a * c_c
    y = tr.f + tr.e * f_c
    lon, lat = rasterio.warp.transform(perfil["crs"], "EPSG:4326",
                                       x.tolist(), y.tolist())
    df["lon_centro"] = np.round(lon, 5)
    df["lat_centro"] = np.round(lat, 5)

    # Celdas sin dominio no aportan informacion: no es que coincidan en
    # cero, es que no habia donde mirar.
    df = df[df.dominio_ha > 0].copy()
    for c in ("propia_ha", "glad_ha", "dominio_ha"):
        df[c] = df[c].round(3)
    return df[["celda", "lon_centro", "lat_centro",
               "propia_ha", "glad_ha", "dominio_ha"]].reset_index(drop=True)


def f1_con_tolerancia(prueba, ref, region, radio: float) -> float:
    """
    F1 aceptando un corrimiento de hasta `radio` pixeles.

    Un pixel marcado por una capa cuenta como acierto si la otra capa
    tiene algun pixel marcado a esa distancia o menos. Asi se separa el
    error de contorno -- el mismo claro con el borde uno o dos pixeles
    mas afuera-- del error de verdad, que es marcar un claro que la otra
    capa no tiene o no ver uno que si tiene.

    radio 1.5 = el anillo de 8 vecinos (30 m); radio 2.9 = dos anillos
    (60 m). Las distancias son euclidianas, por eso la diagonal de un
    pixel mide 1.41 y no 1.
    """
    d_ref = ndimage.distance_transform_edt(~ref)
    d_pru = ndimage.distance_transform_edt(~prueba)
    p, r = prueba & region, ref & region
    if not p.any() or not r.any():
        return 0.0
    prec = float((d_ref[p] <= radio).mean())
    sens = float((d_pru[r] <= radio).mean())
    return 2 * prec * sens / (prec + sens) if prec + sens else 0.0


def referencia_entre_productos(perfil, dominio, region, glad_cod,
                               propia, glad_p) -> pd.DataFrame:
    """
    Cuanto concuerdan entre si productos profesionales, medido igual que
    la demo contra GLAD-L.

    Un F1 por pixel de 0,505 no se puede leer solo: no dice si es bueno o
    malo sin saber cuanto concuerdan, con la misma vara, productos hechos
    por equipos profesionales. Esta tabla da ese punto de referencia.

    Pares, en `region` y con el mismo filtro de parche para todas las
    capas:
      demo vs GLAD-L    ventana de la demo (GLAD_DESDE a GLAD_HASTA)
      Hansen vs GLAD-L  anio calendario ANIO_REFERENCIA
      IDEAM vs GLAD-L   idem, sin los pixeles que el IDEAM marca sin
                        informacion
      IDEAM vs Hansen   idem

    El IDEAM se lee de la capa cambio_(ANIO-1)-(ANIO) que el pipeline ya
    descarga. Si no esta en disco, sus dos filas se omiten.

    COLUMNAS QUE HAY QUE LEER JUNTAS. El F1 por pixel mezcla dos
    desacuerdos distintos: cuanta area reporta cada capa, y donde la
    pone. Si una capa reporta el triple que la otra, el F1 no puede
    pasar de techo_f1 = 2 min(A, B) / (A + B) aunque todo lo que marque
    la chica este dentro de la grande. pct_del_techo dice que parte de
    ese maximo se alcanza. Los productos entre si discrepan sobre todo en
    cuanto; la demo, que se calibro contra GLAD-L, discrepa sobre todo en
    donde. Por eso esta tabla NO permite decir que la demo sea mejor que
    los productos oficiales: dice que su concordancia con GLAD-L esta en
    el mismo rango que la de ellos.
    """
    ha = (ESCALA_M ** 2) / 10_000.0
    anio = ANIO_REFERENCIA
    glad_anio = alertas_glad(glad_cod, dt.date(anio, 1, 1), dt.date(anio, 12, 31))
    perdida = reproyectar_sobre_bloque(
        sorted(DIR_HANSEN.glob("Hansen_*_lossyear_*.tif")), perfil)
    hansen_anio = perdida == (anio - 2000)

    pares = [("demo vs GLAD-L", f"{GLAD_DESDE} a {GLAD_HASTA}",
              propia, glad_p, dominio),
             ("Hansen vs GLAD-L", str(anio), hansen_anio, glad_anio, dominio)]
    capa_ideam = DIR_TRABAJO / f"cambio_{anio - 1}-{anio}.img"
    if capa_ideam.exists():
        ideam = reproyectar_sobre_bloque([capa_ideam], perfil)
        dom_ideam = dominio & np.isin(ideam, [1, 2, 4, 5])
        ideam_anio = ideam == 2
        pares += [("IDEAM vs GLAD-L", str(anio), ideam_anio, glad_anio, dom_ideam),
                  ("IDEAM vs Hansen", str(anio), ideam_anio, hansen_anio, dom_ideam)]
    else:
        logger.info("  %s no esta en disco; se omiten las filas del IDEAM "
                    "(se baja en la sección 5)", capa_ideam.name)

    filas = []
    for par, ventana, prueba, ref, dom in pares:
        prueba = filtrar_parches(prueba & dom)
        ref = filtrar_parches(ref & dom)
        m = matriz_acuerdo(prueba, ref, dom, region)
        mt = metricas(m)
        a_p = float((prueba & region).sum()) * ha
        a_r = float((ref & region).sum()) * ha
        techo = 2 * min(a_p, a_r) / (a_p + a_r) if a_p + a_r else 0.0
        cel = comparar_celdas(prueba, ref, dom, region, perfil)
        r = float(np.corrcoef(cel.propia_ha, cel.glad_ha)[0, 1])
        lin = concordancia_lin(cel.glad_ha, cel.propia_ha)
        det = deteccion_por_parche(prueba, ref, region)
        det = det[det.tamano == "TODOS"].set_index("direccion")
        filas.append({
            "par": par, "ventana": ventana,
            "area_prueba_ha": round(a_p, 1), "area_ref_ha": round(a_r, 1),
            "razon_areas": round(a_p / a_r, 2) if a_r else None,
            "f1": round(mt["f1"], 3),
            "techo_f1": round(techo, 3),
            "pct_del_techo": round(100 * mt["f1"] / techo, 1) if techo else None,
            "f1_tol_1px": round(f1_con_tolerancia(prueba, ref, region, 1.5), 3),
            "f1_tol_2px": round(f1_con_tolerancia(prueba, ref, region, 2.9), 3),
            "pearson_celda": round(r, 3),
            "lin_celda": round(lin, 3),
            "pct_area_ref_hallada": det.loc["glad_vs_propia", "pct_area"],
            "pct_area_prueba_confirmada": det.loc["propia_vs_glad", "pct_area"],
        })
    return pd.DataFrame(filas)

## 14. La muestra para validación visual

50 puntos en cada una de las cuatro combinaciones (los dos marcan, solo la demo, solo GLAD-L, ninguno),
siguiendo a Olofsson et al. (2014). La columna `etiqueta_visual` sale vacía a propósito: la llena una persona
mirando imagen de alta resolución. Hasta que eso pase, la demo mide concordancia con GLAD-L, no exactitud.

In [ ]:
def muestra_estratificada(propia, glad, dominio, region, perfil) -> pd.DataFrame:
    """
    Muestra estratificada sobre las cuatro combinaciones propia/GLAD,
    siguiendo la practica recomendada para estimacion de area y
    exactitud (Olofsson et al., 2014).

    El muestreo es ESTRATIFICADO y no aleatorio simple por una razon
    concreta: las clases de cambio ocupan una fraccion minuscula de la
    escena, asi que una muestra aleatoria simple caeria casi entera en
    bosque estable y dejaria los desacuerdos -- que son justo lo que hay
    que revisar-- con un punado de puntos o ninguno.

    La columna etiqueta_visual sale vacia a proposito: la interpreta un
    humano sobre imagen de alta resolucion, y de ahi salen los errores
    de omision y comision que permiten corregir el area estimada.
    """
    rng = np.random.default_rng(SEMILLA)
    d = dominio & region
    estratos = {
        "ambos": d & propia & glad,
        "solo_propia": d & propia & ~glad,
        "solo_glad": d & ~propia & glad,
        "ninguno": d & ~propia & ~glad,
    }
    tr = perfil["transform"]
    filas = []
    for nombre, m in estratos.items():
        f, c = np.nonzero(m)
        if len(f) == 0:
            logger.warning("  estrato '%s' vacio: no se puede muestrear", nombre)
            continue
        k = min(PUNTOS_POR_ESTRATO, len(f))
        if k < PUNTOS_POR_ESTRATO:
            logger.warning("  estrato '%s': solo %d pixeles disponibles", nombre, k)
        sel = rng.choice(len(f), size=k, replace=False)
        # Centro del pixel, no su esquina.
        x = tr.c + tr.a * (c[sel] + 0.5)
        y = tr.f + tr.e * (f[sel] + 0.5)
        lon, lat = rasterio.warp.transform(perfil["crs"], "EPSG:4326",
                                           x.tolist(), y.tolist())
        filas.append(pd.DataFrame({"lon": np.round(lon, 6),
                                   "lat": np.round(lat, 6),
                                   "estrato": nombre,
                                   "etiqueta_visual": ""}))
    return pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()

## 15. El mapa de revisión

Un mapa con el bosque inicial, lo que quedó sin datos por nubes, la pérdida de la demo y la de GLAD-L, para
revisar cada detección. Las capas van como imágenes sobre el mapa gris claro de Esri, que no pide llave.

In [ ]:
# Mapa base: el gris claro de Esri, que no pide llave. OpenStreetMap y
# CARTO no sirven en BigQuery: el primero bloquea los mapas que no dicen
# desde qué página se piden y el segundo ya exige API key.
TILES_DEFECTO = ("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/"
                 "World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")
ATRIBUCION_TILES = "Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ"


def atenuar_mapa_base(mapa):
    """Mapa base gris y tenue, para que resalten los datos (filtro CSS del navegador)."""
    import folium
    mapa.get_root().header.add_child(folium.Element(
        "<style>.leaflet-tile-pane{filter:grayscale(100%) brightness(107%) contrast(88%);}</style>"))


def _png(ruta: Path, rgb: np.ndarray) -> None:
    """Guarda un array RGBA como PNG, sin dependencias nuevas."""
    import matplotlib.pyplot as plt
    plt.imsave(str(ruta), rgb)


def mapa_revision(capas: Dict[str, np.ndarray], perfil: dict,
                  destino: Path):
    """
    Mapa interactivo con las capas que permiten revisar visualmente cada
    deteccion.

    Se usa folium y no geemap porque el resto del repositorio ya lo usa
    (mapa_folium.py) y porque no exige Earth Engine. Las capas raster se
    superponen como PNG georreferenciados, que es lo que folium admite
    sin un servidor de teselas detras.
    """
    try:
        import folium
        from folium.plugins import Fullscreen
    except ImportError as e:
        logger.warning("No se pudo armar el mapa (%s); se omite.", e)
        return False

    izq, abajo, der, arriba = perfil["bounds"]
    o, s, e, n = transform_bounds(perfil["crs"], "EPSG:4326",
                                  izq, abajo, der, arriba, densify_pts=21)
    limites = [[s, o], [n, e]]

    m = folium.Map(location=[(s + n) / 2, (o + e) / 2], zoom_start=10,
                   tiles=TILES_DEFECTO, attr=ATRIBUCION_TILES, control_scale=True)
    atenuar_mapa_base(m)

    dir_png = destino.parent
    colores = {
        "bosque": (30, 107, 69),        # verde bosque
        "sin_datos": (154, 165, 160),   # gris
        "glad": (59, 53, 166),          # indigo, igual que en el catalogo
        "propia": (184, 86, 45),        # naranja
    }
    encendida = {"propia": True, "glad": True}
    for nombre, color in colores.items():
        m_ = capas[nombre]
        if not m_.any():
            continue
        rgba = np.zeros((*m_.shape, 4), dtype=np.uint8)
        rgba[m_, :3] = color
        rgba[m_, 3] = 200
        ruta = dir_png / f"capa_{nombre}.png"
        _png(ruta, rgba)
        # Se pasa la ruta COMPLETA, no el nombre: folium abre el archivo
        # para incrustarlo en base64 dentro del HTML, y con el nombre
        # suelto lo buscaria en el directorio de trabajo. El HTML queda
        # autonomo, y los PNG sirven ademas para abrirlos en un SIG.
        folium.raster_layers.ImageOverlay(
            image=str(ruta), bounds=limites, opacity=0.75,
            name={"bosque": "Bosque inicial", "sin_datos": "Sin datos (nubes)",
                  "glad": "Perdida GLAD-L", "propia": "Perdida propia"}[nombre],
            show=encendida.get(nombre, False)).add_to(m)

    folium.Rectangle(limites, color="#16211C", weight=2, fill=False,
                     name="Zona piloto").add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    Fullscreen(position="topleft").add_to(m)
    m.save(str(destino))
    return m

## 16. La corrida, paso a paso

Primero los insumos (bosque, escenas, GLAD-L), porque cualquiera puede faltar y descubrirlo al final costaría
toda la corrida; después el diagnóstico de nubes, que define el dominio; y solo entonces calibración,
validación y muestreo. Cada resultado se muestra aquí y se guarda en `DATASET_RESULTADOS`.

In [ ]:
def guardar_demo(df, nombre, descripcion, indice=False):
    """Guarda un resultado de la demo en DATASET_RESULTADOS, como demo_landsat_<nombre>."""
    guardar_tabla(df, f"demo_landsat_{nombre}", "Demo de Landsat (notebook 18): " + descripcion,
                  dataset=DATASET_RESULTADOS, indice=indice)


perfil = perfil_zona()
oeste, este = mascara_mitades(perfil)
ha = (ESCALA_M ** 2) / 10_000.0
print(f"Zona piloto {BBOX}: rejilla de {perfil['width']} x {perfil['height']} píxeles de {ESCALA_M} m ({CRS})")
bosque = mascara_bosque(perfil)
print(f"Bosque inicial: {int(bosque.sum()):,} píxeles = {bosque.sum() * ha:,.1f} ha")

### 16.1 Los compuestos de Landsat

Es la parte lenta: lee por internet el recorte de unas 70 escenas.

In [ ]:
nbr_t1, obs_t1, des_t1, nbr_t2, obs_t2, des_t2 = compuestos_con_cache(perfil)
dnbr = nbr_t1 - nbr_t2

### 16.2 El dominio y el diagnóstico de nubes

`sin_datos_pct` dice qué parte del bosque quedó fuera por no tener ninguna lectura limpia; las columnas
`obs_*` dicen con cuántas lecturas limpias quedó cada píxel. En el trabajo de grado, de 38 escenas el píxel
mediano tuvo 5 limpias.

In [ ]:
dominio = construir_dominio(bosque, obs_t1, obs_t2)
if dominio.sum() == 0:
    raise RuntimeError("El dominio quedó vacío: no hay bosque con observaciones limpias en ambas ventanas.")
diag = diagnostico_nubes(bosque, dominio, obs_t1, obs_t2)
guardar_demo(diag, "diagnostico_nubes", "bosque observable y número de lecturas limpias por píxel.")
diag.T

### 16.3 Qué fecha representa cada compuesto

In [ ]:
fe = fecha_efectiva(dominio, des_t1, des_t2).set_index("medida")
guardar_demo(fe, "fecha_efectiva", "qué fecha representa cada compuesto y qué intervalo mide el dNBR.", indice=True)
fe

### 16.4 GLAD-L

In [ ]:
glad_cod = leer_glad(perfil, _api_key())
glad_bruta = alertas_glad(glad_cod)
glad_p = perdida_glad(glad_bruta, dominio)
print(f"GLAD-L en la ventana y en el dominio: {glad_p.sum() * ha:,.1f} ha")

### 16.5 Calibración del umbral, en la mitad oeste

El F1 debe subir, hacer pico y bajar: así el umbral elegido es un óptimo y no el borde de la búsqueda.

In [ ]:
cal = calibrar_umbral(dnbr, dominio, glad_p, oeste)
guardar_demo(cal, "calibracion_umbral", "métricas de cada umbral de dNBR en la mitad oeste (calibración).")
mejor = float(cal.loc[cal.f1.idxmax(), "umbral_dnbr"])
print(f"Umbral elegido por F1: {mejor:.2f}")
cal

### 16.6 Validación, en la mitad este

In [ ]:
propia = perdida_propia(dnbr, dominio, mejor)
m = matriz_acuerdo(propia, glad_p, dominio, este)
ev = pd.DataFrame([{"umbral_dnbr": mejor, **m, **metricas(m)}])
guardar_demo(ev, "evaluacion_validacion", "matriz de acuerdo y métricas en la mitad este (validación).")
ev.T

### 16.7 Concordancia según cuántas observaciones se exijan

Ojo con las filas exigentes: en el sur de la zona se traslapan dos órbitas de Landsat, y pedir muchas
observaciones deja solo el sur (`reparto_ns` cae a cero). Esas filas describen una región, no la zona.

In [ ]:
curva = curva_observabilidad(lambda dom, u: perdida_propia(dnbr, dom, u),
                             glad_bruta, bosque, obs_t1, obs_t2, este, mejor)
guardar_demo(curva, "curva_observabilidad", "concordancia con GLAD-L según las observaciones limpias exigidas.")
curva

### 16.8 Detección por claro

In [ ]:
par = deteccion_por_parche(propia, glad_p, este)
guardar_demo(par, "deteccion_por_parche", "claros detectados por tramo de tamaño, en las dos direcciones.")
par

### 16.9 Comparación por celda de 5 km

In [ ]:
celdas = comparar_celdas(propia, glad_p, dominio, este, perfil)
guardar_demo(celdas, "comparacion_celdas", "hectáreas de pérdida por celda de 5 km, según la demo y según GLAD-L.")
if len(celdas) >= 3:
    print(f"Celdas con dominio: {len(celdas)}")
    print(f"Lin      = {concordancia_lin(celdas.glad_ha, celdas.propia_ha):.3f}  (concordancia: dónde y cuánto)")
    print(f"Pearson  = {celdas.propia_ha.corr(celdas.glad_ha, method='pearson'):.3f}  (solo si suben y bajan juntas)")
    print(f"Spearman = {celdas.propia_ha.corr(celdas.glad_ha, method='spearman'):.3f}")
else:
    print(f"Muy pocas celdas ({len(celdas)}) para correlacionar.")

### 16.10 Referencia: cuánto concuerdan entre sí los productos

In [ ]:
ref = referencia_entre_productos(perfil, dominio, este, glad_cod, propia, glad_p)
guardar_demo(ref, "referencia_entre_productos",
             "concordancia entre la demo, GLAD-L, Hansen y el IDEAM, medida con la misma vara.")
ref

### 16.11 Muestra para validación visual

In [ ]:
muestra_visual = muestra_estratificada(propia, glad_p, dominio, este, perfil)
guardar_demo(muestra_visual, "muestra_validacion_visual",
             "muestra estratificada de puntos para interpretar a mano (columna etiqueta_visual vacía).")
muestra_visual.estrato.value_counts()

### 16.12 Mapa de revisión

In [ ]:
mapa = mapa_revision({"bosque": bosque, "sin_datos": bosque & ~dominio, "propia": propia, "glad": glad_p},
                     perfil, DIR_DEMO / "mapa_revision.html")
mapa

## 17. Lo que queda en BigQuery

En `DATASET_RESULTADOS`:

| Tabla | Qué es |
|---|---|
| `DEMO_LANDSAT_DIAGNOSTICO_NUBES` | Bosque observable y lecturas limpias por píxel |
| `DEMO_LANDSAT_FECHA_EFECTIVA` | Qué fecha representa cada compuesto y qué intervalo mide el dNBR |
| `DEMO_LANDSAT_CALIBRACION_UMBRAL` | Métricas de cada umbral, en la mitad oeste |
| `DEMO_LANDSAT_EVALUACION_VALIDACION` | Matriz de acuerdo y métricas, en la mitad este |
| `DEMO_LANDSAT_CURVA_OBSERVABILIDAD` | Concordancia según las observaciones exigidas |
| `DEMO_LANDSAT_DETECCION_POR_PARCHE` | Claros detectados por tramo de tamaño |
| `DEMO_LANDSAT_COMPARACION_CELDAS` | Hectáreas por celda de 5 km, demo y GLAD-L |
| `DEMO_LANDSAT_REFERENCIA_ENTRE_PRODUCTOS` | Concordancia entre productos con la misma vara |
| `DEMO_LANDSAT_MUESTRA_VALIDACION_VISUAL` | Puntos para la validación visual |

## 18. Cómo leer los resultados

- **A nivel de píxel la coincidencia es moderada; a la escala de 5 km es alta.** Los dos métodos discrepan
  sobre qué píxel exacto marcar y coinciden sobre dónde está ocurriendo la pérdida y cuánta hay, que es lo que
  necesita un modelo que trabaja por celda. Un claro detectado por los dos con el borde corrido uno o dos
  píxeles castiga la precisión por píxel sin cambiar el total de la celda.
- **Lo que más pesa en los desacuerdos son las nubes residuales**, los bordes y los claros pequeños, no la
  fecha de los compuestos.
- **El F1 está en el rango de los productos profesionales entre sí** (tabla de referencia), lo que no quiere
  decir que la demo sea mejor que ellos: su concordancia con GLAD-L está en el mismo rango que la de ellos.

Los límites de la demo y las preguntas que deja abiertas están en `guias/4_DEMO_LANDSAT.md`.